# SPACEc: Preprocessing - Fluorescence spillover compensation (optional)

Lateral spillover correction is a post-segmentation strategy designed to refine single-cell fluorescence measurements in densely packed tissues. In multiplexed imaging, the fluorescence from a given cell can inadvertently extend into neighboring cells—a phenomenon often described as "bleed" or "spillover." This method first quantifies the extent of shared contact between adjacent cells and then adjusts the fluorescence intensities by subtracting the estimated contaminating signal. In doing so, it reassigns the proper fluorescence to each cell, ensuring that the measured protein expression more accurately reflects each cell’s true biology.

The correction is expressed as one linear system per channel, whose coefficients are the shared-boundary fractions between touching cells. Because a cell only ever touches a handful of neighbors, that system is sparse, and SPACEc solves it directly on the CPU — a whole region runs in seconds without a GPU.

The pillover compensation procedure is further explained in these two publications:
1) Lee, M. Y., Bedia, J. S., Bhate, S. S., Barlow, G. L., Phillips, D., Fantl, W. J., Nolan, G. P., & Schürch, C. M. (2022). CellSeg: a robust, pre-trained nucleus segmentation and pixel quantification software for highly multiplexed fluorescence images. BMC Bioinformatics, 23(1). https://doi.org/10.1186/s12859-022-04570-9
2) Goltsev, Y., Samusik, N., Kennedy-Darling, J., Bhate, S., Hale, M., Vazquez, G., Black, S., & Nolan, G. P. (2018). Deep Profiling of Mouse Splenic Architecture with CODEX Multiplexed Imaging. Cell, 174(4), 968-981.e15. https://doi.org/10.1016/j.cell.2018.07.010

For the purpose of this tutorial we will use the output of the Cellpose segmentation (Mesmer or other results can be used as well).

The steps of the script are:

1. Loading each region's mask, images, and features table
2. Running spillover correction for multiplexed image
3. Visualize the compensation results for an example image
4. Store the data for further processing

In [ ]:
# import spacec first
import spacec as sp

#import standard packages
import os
import pandas as pd

In [ ]:
# Apply SPACEc publication style to all plots in this notebook
sp.pl.set_style(sp.pl.SPACEcStyle.publication())

In [ ]:
# Specify the path to the data
data_path = "path/to/example_data/raw/"
output_dir = "path/to/output/"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
sp.hf.check_for_gpu()

## Load data
Load each region's mask and channel images by path (written by `cell_segmentation(save_mask=True)` in 02_cell_segmentation -- no `seg_output` pickle needed), and the features table via `read_segdf` so it carries the `label`/`unique_region` columns `compensate_cell_matrix` joins on.

In [ ]:
# Each region's raw label mask, written by cell_segmentation(save_mask=True) in 02_cell_segmentation
masks = sp.hf.hf_load_masks({
    "reg001": output_dir + "tonsil1_mask.tif",
    "reg002": output_dir + "tonsil2_mask.tif",
})

# Each region's channel images, reloaded from the same multichannel tif segmented in 02_cell_segmentation
image_dict1 = sp.hf.hf_load_image_dict(
    file_name = output_dir + "reg001_X01_Y01_Z01.tif",
    channel_file = data_path + "channelnames.txt",
    input_format = "Multichannel",
    nuclei_channel = "DAPI",
)
image_dict2 = sp.hf.hf_load_image_dict(
    file_name = output_dir + "reg002_X01_Y01_Z01.tif",
    channel_file = data_path + "channelnames.txt",
    input_format = "Multichannel",
    nuclei_channel = "DAPI",
)

# Load both regions' features via read_segdf (not a raw pd.read_csv) -- this is the
# same object 03.1 works with, carrying the 'label'/'unique_region' columns
# compensate_cell_matrix needs to join compensated values back onto by mask label.
df_seg = sp.pp.read_segdf(
    segfile_list = [output_dir + "tonsil1_features.csv", output_dir + "tonsil2_features.csv"],
    seg_method = "cellpose",
    region_list = ["reg001", "reg002"],
)

## Run compensation
Compensate each region separately -- mask labels are only unique *within* a region, so `compensate_cell_matrix` requires `region=...` and joins compensated values back onto `df_seg` by that region's `label` column. We chain the two calls onto the same `df_seg` so the result ends up as one combined table with both regions' compensated columns filled in.

Compensation runs on the CPU and needs memory proportional to the number of cells, not its square, so memory use stays modest: a 120,000-cell region takes a few seconds and about 10 MB. The channel images themselves are the only large thing in memory, which is why this notebook loads one region's `image_dict` at a time.

`overwrite=False` keeps the raw marker columns and adds `<marker>_compensated` columns alongside them, so the effect of compensation can be inspected below. Once you're satisfied with the result, run this same step inside 03.1 with `overwrite=True` instead, which replaces the raw columns in place and is what the rest of the pipeline (filtering, normalization) should see.

In [ ]:
df_c = sp.pp.compensate_cell_matrix(
    df_seg,
    image_dict = image_dict1, # region 1's channel images
    masks = masks["reg001"], # region 1's raw label mask
    region = "reg001", # mask labels are only unique within a region
    overwrite = False, # keep raw columns, add '<marker>_compensated' for before/after comparison
)

In [ ]:
df_c = sp.pp.compensate_cell_matrix(
    df_c,
    image_dict = image_dict2, # region 2's channel images
    masks = masks["reg002"], # region 2's raw label mask
    region = "reg002", # mask labels are only unique within a region
    overwrite = False, # keep raw columns, add '<marker>_compensated' for before/after comparison
)

# a single-region view for the visualizations below, which plot spatial (x, y)
# coordinates that are only meaningful within one region
region2 = df_c[df_c["unique_region"] == "reg002"]

## Visualize compensation results

Visualze example marker in tissue.

In [ ]:
sp.pl.compensation_qc(region2, markers="CD20")

`compensation_qc` can optionally fill each cell's actual segmentation-mask footprint instead of drawing a dot at its centroid -- pass the region's label mask via `masks=`. This is especially useful for seeing spillover between touching cells at high density, where overlapping dots can obscure exactly the effect this QC plot is meant to show.

In [ ]:
sp.pl.compensation_qc(region2, markers="CD20", masks=masks["reg002"])

Zoom in on an image part to inspect compensation results in more detail.

In [ ]:
# Define a collection of biomarkers of interest for visualization
biomarkers_of_interest = ['CD20', 'CD3', 'FoxP3', 'HLA-DR', 'EGFR']

In [ ]:
# Define the zoom-in region.
x_min, x_max = 1000, 2250
y_min, y_max = 1500, 2750

In [ ]:
# Visualize the compensated values for the biomarkers of interest, zoomed into the region defined above.
subset = region2[(region2['x'] >= x_min) & (region2['x'] <= x_max) &
                 (region2['y'] >= y_min) & (region2['y'] <= y_max)]

sp.pl.compensation_qc(subset, markers=biomarkers_of_interest, point_size=35)

Mask-fill mode also works on a spatially-cropped subset like this one -- pass the same full-region `masks` array and let the axis limits handle the zoom, no need to crop the mask array to match (`point_size` is ignored in this mode, since there are no dots to size).

In [ ]:
sp.pl.compensation_qc(subset, markers=biomarkers_of_interest, masks=masks["reg002"])

## When compensation helps -- and when it doesn't
Spillover compensation is most useful in **densely packed tissue** with **high spillover** -- small, tightly touching cells, or markers known to bleed between neighbors (e.g. a strong membrane marker leaking into an adjacent cell's cytoplasm signal). It's less useful, and can even slightly blur real signal, in **sparse tissue** or on **already-clean panels** where cells rarely touch -- compare the before/after `compensation_qc` plots above for your own data before deciding whether to apply it.

Once you're satisfied with the result, run this same step inside [03.1_preprocessing.ipynb](03.1_preprocessing.ipynb) with `overwrite=True` right after `read_segdf` -- that replaces the raw marker columns in place with the compensated values, which is what filtering and normalization downstream should see. There's no separate compensated file to save from this notebook: it exists to let you inspect the effect of compensation before committing to it in 03.1.